# Latent Text Compressor playground

Choose your configuration, optionally prepare/train, then inspect exact recovery. The smoke recipe only checks the workflow. Full recovery needs a trained checkpoint. No weights are downloaded.


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

from latent_text.runtime import read_config
from latent_text.codec import Codec

CONFIG = "config/train.json"
CHECKPOINT = Path("artifacts/runs/paragraph/span32/last.pt")
DEVICE = "auto"
recipe = read_config(CONFIG)
recipe


## Optional: prepare and train

Enable these switches deliberately. Preparation streams the pinned corpus. Training runs all stages. For a small installation check choose `config/smoke.json` above and use the CLI smoke preparation command in the README.


In [ ]:
RUN_PREPARATION = False
RUN_TRAINING = False

if RUN_PREPARATION:
    from latent_text.data import prepare
    prepare(recipe["data_dir"], **recipe["dataset"])

if RUN_TRAINING:
    from latent_text.train import train
    CHECKPOINT = train(recipe)


## Load a trained checkpoint

Set `CHECKPOINT` to your own local model. The decoder predicts all original positions in parallel.


In [ ]:
codec = Codec.load(CHECKPOINT, DEVICE) if CHECKPOINT.is_file() else None
if codec is None:
    print("Train first or select your existing checkpoint.")
else:
    print("Parameters:", sum(p.numel() for p in codec.model.parameters()))
    print("Tokens per vector:", codec.model.config.span)


In [ ]:
TEXT = "Invoice 17019: 17.09, not 19.07."

if codec is not None:
    result = codec.reconstruct(TEXT)
    print("INPUT: ", repr(TEXT))
    print("OUTPUT:", repr(result["output"]))
    print("Exact:", result["exact_match"])
    print({key: value for key, value in result.items() if key not in ("input", "output")})


## Decode saved vectors alone

The second operation below receives only the saved payload and the matching codec. It does not receive `TEXT`. Saved files remain under ignored `artifacts/`.


In [ ]:
if codec is not None:
    path = codec.save_memory(TEXT, "artifacts/notebook-memory.pt")
    recovered = codec.recover_file(path)
    print("FROM VECTORS:", repr(recovered))
    print("Exact:", recovered == TEXT)
    print("Serialized file bytes:", path.stat().st_size)
